In [1]:
# Auto-generated by cram CLI — do not hand-edit.
# Rebuild with: python tools/cram.py build

import sys
from pathlib import Path

# Portably discover repo root containing 'engine' and add to sys.path
_cur = Path.cwd().resolve()
while _cur != _cur.parent:
    if (_cur / 'engine' / '__init__.py').is_file():
        if str(_cur) not in sys.path:
            sys.path.insert(0, str(_cur))
        break
    _cur = _cur.parent

from engine import Worksheet, apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_08_quantum_mechanics_and_dft.yaml')
ws.start()


$\def\AA{\text{Å}}$

# MSE 485 · Lesson 08: Quantum Mechanics, Many-Electron Wavefunctions, and Density Functional Theory

_Session 7dd24bd9 · Estimated time: ~135 minutes · 46 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Contrast the accuracy-vs-speed hierarchy of atomistic energy models and solve the Schrödinger equation for canonical 1D and 3D radial potentials.
- Apply the Rayleigh-Ritz Variational Principle and derive the complete ground-state energy minimization for the Hydrogen atom using a trial exponential wavefunction.
- Quantify the exponential curse of dimensionality for many-electron wavefunctions and compare the Hartree product, Hartree-Fock Slater determinants, Pauli exchange, and electron correlation.
- Formulate Density Functional Theory via the 1st and 2nd Hohenberg-Kohn theorems, the Kohn-Sham auxiliary equations, and Jacob's Ladder of exchange-correlation functionals (LDA, GGA, Meta-GGA, Hybrids).
- Explain norm-conserving pseudopotentials, plane-wave basis cutoffs, Bloch's theorem, Brillouin-zone Bragg band gaps, and the DFT semiconductor band-gap problem.


In [2]:
ws.render_section('sec_hierarchy_and_schrodinger')

## 1. Hierarchy of Energy Models & The Schrödinger Equation

In atomic-scale simulations (`05.QuantMech.pdf`, pp. 1–11), choosing an **energy model** $E(\{\mathbf{R}_I\})$ involves a fundamental trade-off between **accuracy/transferability** and **computational speed**:
- **Empirical Potentials** (Lennard-Jones, EAM, OPLS): fastest ($\sim 10^6\text{–}10^9$ atoms, molecular dynamics), fitted to data, no explicit electrons.
- **Semi-Empirical / Tight-Binding Models**: intermediate speed and accuracy, parameterized quantum hopping integrals.
- **Quantum Mechanical First-Principles (*Ab Initio*) Methods** (Density Functional Theory, Hartree–Fock, Configuration Interaction, Quantum Monte Carlo): highest accuracy, capable of predicting chemical bonding, magnetism, optical excitations, and transition states directly from atomic numbers $Z_I$.

In the **Standard Model of Matter** for materials science, atoms consist of massive, point-like nuclei (protons + neutrons) surrounded by **tightly bound core electrons** and held together in molecules, liquids, and solids by chemical bonds formed by **valence electrons**.

### Wave–Particle Duality & The Schrödinger Equation
By de Broglie / Planck wave–particle duality ($\lambda \cdot p = h$, or $\mathbf{p} = \hbar\mathbf{k}$), electrons obey the **time-dependent Schrödinger equation**:
$$-\frac{\hbar^2}{2m}\nabla^2 \Psi(\mathbf{r}, t) + V(\mathbf{r}, t)\Psi(\mathbf{r}, t) = i\hbar\frac{\partial \Psi(\mathbf{r}, t)}{\partial t}$$
- **Mathematical Characteristics (Slide 4 Exercise):** This is a **linear**, **homogeneous** partial differential equation that is **first-order in time $t$** and **second-order in space $\mathbf{r}$**, with complex-valued solutions $\Psi(\mathbf{r},t)$ whose squared modulus $|\Psi(\mathbf{r},t)|^2$ gives the Born probability density.
- **Separation of Variables (Slide 5 Exercise):** When the potential is time-independent, $V(\mathbf{r}, t) = V(\mathbf{r})$, substituting $\Psi(\mathbf{r}, t) = \varphi(\mathbf{r})\cdot f(t)$ yields $f(t) = e^{-iEt/\hbar}$ and the **time-independent Schrödinger eigenvalue equation** $\hat{\mathcal{H}}\varphi(\mathbf{r}) = E\varphi(\mathbf{r})$:
  $$\left[-\frac{\hbar^2}{2m}\nabla^2 + V(\mathbf{r})\right]\varphi(\mathbf{r}) = E\,\varphi(\mathbf{r})$$

### Canonical Analytical Solutions (Slides 6–9)
1. **Free Particle ($V(\mathbf{r}) = 0$):** Plane waves $\Psi_{\mathbf{k}}(\mathbf{r},t) = \exp(i\mathbf{k}\cdot\mathbf{r} - i\omega t)$ with continuous parabolic energy dispersion $E_k = \frac{\hbar^2 k^2}{2m} = \hbar\omega$.
2. **1D Harmonic Oscillator ($V(x) = \frac{1}{2}kx^2$):** Gaussian times Hermite polynomials $\varphi_n(x) = c_n \exp\left(-\frac{m\omega^2 x^2}{2\hbar\omega}\right)H_n\left(\sqrt{\frac{m\omega}{\hbar}}x\right)$ with equally spaced quantized levels $E_n = \left(n + \frac{1}{2}\right)\hbar\omega$ for $n = 0, 1, 2, \dots$ (note the non-zero **zero-point energy** $E_0 = \frac{1}{2}\hbar\omega$).
3. **1D Infinite Square Well ($V(x) = 0$ for $0 < x < a$, $\infty$ otherwise):** Standing sine waves vanishing at the boundaries, $\varphi_n(x) = \sqrt{\frac{2}{a}}\sin\left(\frac{\pi n x}{a}\right)$, with quadratic quantum-confinement energies $E_n = \frac{\hbar^2 \pi^2 n^2}{2 m a^2} \propto \frac{n^2}{a^2}$ ($n = 1, 2, 3, \dots$).
4. **Metal Surface Potential Step ($V(x) = 0$ inside metal $x < 0$, $V(x) = V_0 > E$ in vacuum $x > 0$):** Oscillatory plane wave inside the metal that **decays exponentially** ($\psi(x) \propto e^{-\kappa x}$ with $\kappa = \sqrt{2m(V_0 - E)}/\hbar$) into the vacuum outside the surface (creating the surface dipole and work function, and enabling Scanning Tunneling Microscopy!).
5. **Spherical Coulomb Potential ($V(r) = -\frac{Ze^2}{4\pi\varepsilon_0 r}$):** Separating variables in spherical coordinates $\Psi_{nlm}(\mathbf{r}) = R_{nl}(r)Y_{lm}(\theta, \varphi)$ yields the radial equation with the **centrifugal barrier** $\frac{l(l+1)\hbar^2}{2mr^2}$:
   $$\left[-\frac{\hbar^2}{2m}\left(\frac{d^2}{dr^2} + \frac{2}{r}\frac{d}{dr}\right) + \frac{l(l+1)\hbar^2}{2mr^2} + V(r)\right]R_{nl}(r) = E\,R_{nl}(r)$$

### Matrix Formulation in an Orthogonal Basis (Slide 11)
Expanding $|\psi\rangle = \sum_{n=1}^k c_n |\varphi_n\rangle$ in an orthonormal basis ($\langle\varphi_m|\varphi_n\rangle = \delta_{mn}$) and left-multiplying $\hat{\mathcal{H}}|\psi\rangle = E|\psi\rangle$ by $\langle\varphi_m|$ converts the differential equation into an algebraic $k \times k$ **matrix eigenvalue problem**:
$$\sum_{n=1}^k H_{mn} c_n = E c_m \quad \Longleftrightarrow \quad \mathbf{H}\mathbf{c} = E\mathbf{c}, \qquad \text{where } H_{mn} = \langle\varphi_m|\hat{\mathcal{H}}|\varphi_n\rangle$$


In [3]:
ws.render_section('q01_hierarchy_of_models')

**Q1 · easy** · 💡

In the hierarchy of atomistic energy models (`05.QuantMech.pdf`, pp. 1–2), which class of electrons actually forms chemical bonds holding molecules, liquids, and solids together, and why must we use quantum mechanics rather than classical empirical force fields to model bond breaking, magnetism, or optical excitations?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Contrast core electrons vs. valence electrons on Slide 2, and compare what empirical models lack on Slide 1.

</div>
</details>

- [ ] **(A)** Valence electrons form chemical bonds (while core electrons remain tightly bound to nuclei); empirical potentials pre-assume fixed bonding or fitted pair functions and lack explicit electronic wavefunctions/densities needed for bond rearrangement, magnetism, and optical transitions.
- [ ] **(B)** Core 1s electrons dominate chemical bonding in all transition metals, while valence electrons remain inert.
- [ ] **(C)** Empirical potentials solve the time-dependent Schrödinger equation exactly but are slower than DFT.
- [ ] **(D)** Quantum mechanical methods only calculate nuclear kinetic energy and ignore electrons.

In [4]:
ws.render_section('q02_schrodinger_pde_properties')

**Q2 · easy** · 💡

Which of the following are true mathematical characteristics of the time-dependent Schrödinger equation $-\frac{\hbar^2}{2m}\nabla^2\Psi(\mathbf{r},t) + V(\mathbf{r},t)\Psi(\mathbf{r},t) = i\hbar\frac{\partial\Psi(\mathbf{r},t)}{\partial t}$ (Slide 4 Exercise)? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the order of spatial and temporal derivatives, whether Psi appears to the 1st power (linear), and whether any term is independent of Psi (homogeneous).

</div>
</details>

- [ ] **(A)** It is a linear and homogeneous partial differential equation (any linear combination of solutions is also a solution, enabling quantum superposition).
- [ ] **(B)** It is second-order in spatial coordinates (nabla^2) and first-order in time (d/dt).
- [ ] **(C)** For a time-independent potential V(r), separation of variables Psi(r,t) = phi(r) * f(t) yields stationary states with harmonic time evolution f(t) = exp(-i*E*t/hbar).
- [ ] **(D)** It is a quadratic non-linear equation in Psi(r,t).

In [5]:
ws.render_section('q03_infinite_square_well_ratio_numeric')

**Q3 · easy** · 💡

For an electron in a 1D infinite square well of width $a$ (`05.QuantMech.pdf`, p. 7), the quantized energy levels are $E_n = \frac{\hbar^2 \pi^2 n^2}{2ma^2}$ for $n = 1, 2, 3, \dots$. If the ground-state energy ($n=1$) is $E_1 = 1.50\text{ eV}$, what is the energy $E_4$ of the $n = 4$ state (in $\text{eV}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Notice from Slide 7 that E_n / E_1 = n^2. For n = 4, multiply E_1 by 4^2 = 16.

</div>
</details>

_Numeric answer required._

In [6]:
ws.render_section('q04_harmonic_oscillator_vs_square_well')

**Q4 · medium** · 💡

Compare the energy level spacing $\Delta E_n = E_{n+1} - E_n$ of a 1D quantum harmonic oscillator ($V(x) = \frac{1}{2}kx^2$, Slide 6) against that of a 1D infinite square well ($V(x) = 0$ on $(0,a)$, Slide 7). How do their level spacings behave as the quantum number $n$ increases?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare E_n = (n + 1/2)*hbar*omega vs. E_n = E_1 * n^2.

</div>
</details>

- [ ] **(A)** The harmonic oscillator has constant level spacing Delta E_n = hbar*omega for all n (with zero-point energy E_0 = 1/2 hbar*omega), whereas the infinite square well spacing grows linearly as Delta E_n = (2n + 1)*E_1.
- [ ] **(B)** Both potentials have constant, equally spaced energy levels.
- [ ] **(C)** The harmonic oscillator level spacing grows as n^2, whereas the square well spacing decreases as 1/n^2.
- [ ] **(D)** The harmonic oscillator has zero ground-state energy E_0 = 0.

In [7]:
ws.render_section('q05_metal_surface_step_wavefunction')

**Q5 · easy** · 💡

According to Slide 8 (`05.QuantMech.pdf`), how does a conduction electron wavefunction $\psi(x)$ with energy $E < V_0$ behave across a metal surface modeled as a potential step ($V(x) = 0$ inside the metal $x < 0$, and $V(x) = V_0$ in the vacuum $x > 0$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Examine the plot of psi(y) on Slide 8 across the surface boundary at 0.

</div>
</details>

- [ ] **(A)** It is an oscillatory plane/standing wave inside the metal (x < 0) and decays exponentially into the vacuum outside the metal (x > 0).
- [ ] **(B)** It vanishes discontinuously to zero at x = 0 with infinite slope.
- [ ] **(C)** It decays exponentially inside the metal and oscillates with constant amplitude in the vacuum.
- [ ] **(D)** It grows exponentially to infinity as x -> +infinity in the vacuum.

In [8]:
ws.render_section('q06_radial_coulomb_centrifugal_barrier')

**Q6 · medium** · 💡

In the radial Schrödinger equation for a spherically symmetric Coulomb potential (Slide 9),
$$\left[-\frac{\hbar^2}{2m}\left(\frac{d^2}{dr^2} + \frac{2}{r}\frac{d}{dr}\right) + \frac{l(l+1)\hbar^2}{2mr^2} + V(r)\right]R_{nl}(r) = E\,R_{nl}(r)$$
what is the physical origin and effect of the term $\frac{l(l+1)\hbar^2}{2mr^2}$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall that the angular momentum squared operator L^2 has eigenvalues l(l+1)*hbar^2 on spherical harmonics Y_{lm}(theta, phi).

</div>
</details>

- [ ] **(A)** It is the repulsive centrifugal barrier from orbital angular momentum L^2 = l(l+1)*hbar^2, which pushes higher-l states (p, d, f with l >= 1) away from the nucleus at r -> 0 while vanishing for s-states (l = 0).
- [ ] **(B)** It is an attractive magnetic spin-orbit coupling term that pulls d and f electrons into the nucleus.
- [ ] **(C)** It is the electron-electron Hartree repulsion term.
- [ ] **(D)** It is non-zero only for s-orbitals (l = 0).

In [9]:
ws.render_section('q07_hydrogen_quantum_numbers_degeneracy_numeric')

**Q7 · medium** · 💡

For a hydrogenic atom (`05.QuantMech.pdf`, pp. 9–10), the principal quantum number $n$ allows orbital angular momentum quantum numbers $l = 0, 1, \dots, n-1$, and each $l$ has $2l + 1$ magnetic sublevels $m \in \{-l, \dots, +l\}$. For the $n = 3$ shell ($3s, 3p, 3d$), how many total spatial orbitals $(l, m)$ exist (excluding spin)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Sum (2l + 1) for l = 0, 1, 2 (which equals 1 + 3 + 5 = n^2).

</div>
</details>

_Numeric answer required._

In [10]:
ws.render_section('q08_matrix_formulation_basis_size')

**Q8 · medium** · 💡

In the matrix formulation of quantum mechanics (Slide 11), expanding the wavefunction in $k$ orthonormal basis functions $|\psi\rangle = \sum_{n=1}^k c_n|\varphi_n\rangle$ transforms $\hat{\mathcal{H}}|\psi\rangle = E|\psi\rangle$ into $\sum_{n=1}^k H_{mn}c_n = E c_m$. How are the matrix elements $H_{mn}$ defined, and how does the cost of standard dense matrix diagonalization scale with the basis size $k$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check Slide 11 for the bra-ket definition of H_{mn}, and recall the asymptotic complexity of dense eigensolvers.

</div>
</details>

- [ ] **(A)** H_{mn} = <phi_m | H | phi_n> is the Hamiltonian integral between basis states m and n, and dense matrix diagonalization scales as O(k^3).
- [ ] **(B)** H_{mn} = <phi_m | phi_n> is the identity matrix, and diagonalization scales as O(1).
- [ ] **(C)** H_{mn} = c_m * c_n, and diagonalization scales linearly as O(k).
- [ ] **(D)** H_{mn} = E * delta_{mn} for any arbitrary non-eigenstate basis.

In [11]:
ws.render_section('sec_variational_principle_and_hydrogen')

## 2. The Variational Principle & Worked Derivation for the Hydrogen Atom

When we cannot solve $\hat{\mathcal{H}}\Psi = E\Psi$ analytically, the **Rayleigh–Ritz Variational Principle** (`05.QuantMech.pdf`, pp. 12–15) provides the mathematical foundation for almost every computational electronic-structure method:
$$\boxed{E[\Phi] = \frac{\langle\Phi|\hat{\mathcal{H}}|\Phi\rangle}{\langle\Phi|\Phi\rangle} \ge E_0}$$
- For **any** normalizable trial wavefunction $\Phi$, the energy expectation value $E[\Phi]$ is an **upper bound** to the true ground-state energy $E_0$.
- Equality holds ($E[\Phi] = E_0$) **if and only if** $\Phi$ is the exact ground-state wavefunction $\Psi_0$.
- Therefore, if we parameterize a family of trial wavefunctions $\Phi_\alpha$ and minimize $E(\alpha)$ with respect to $\alpha$ ($\frac{dE}{d\alpha} = 0$), we obtain the closest possible approximation to the ground state within that family!

### Atomic Units (Slide 15)
To eliminate fundamental constants from quantum mechanical equations, we set **$\hbar = m_e = e = \frac{1}{4\pi\varepsilon_0} = 1$**:
- **Length:** $1\text{ Bohr radius } (a_0) = 5.29177 \times 10^{-11}\text{ m} = 0.529177\text{ Å}$
- **Energy:** $1\text{ Hartree } (\text{Ha}) = \frac{e^2}{4\pi\varepsilon_0 a_0} = 4.35974 \times 10^{-18}\text{ J} = 27.2114\text{ eV}$ (note that $1\text{ Ry} = 0.5\text{ Ha} = 13.6057\text{ eV}$)

### Complete Variational Derivation for the Hydrogen Atom (Slides 12–14)
In atomic units, the Hydrogen Hamiltonian is $\hat{\mathcal{H}} = -\frac{1}{2}\nabla^2 - \frac{1}{r}$. Let us test the spherically symmetric exponential **Ansatz**:
$$\Phi_\alpha(r) = c \cdot \exp(-\alpha r)$$
Using the spherical volume element $\langle \dots | \dots \rangle = \int_0^\infty 4\pi r^2 (\dots)\,dr$, the standard radial Gamma integral $\int_0^\infty r^n e^{-ar}\,dr = \frac{n!}{a^{n+1}}$, and the radial Laplacian $\nabla^2\Phi_\alpha = \frac{1}{r^2}\frac{d}{dr}\left(r^2\frac{d\Phi_\alpha}{dr}\right) = \left(\alpha^2 - \frac{2\alpha}{r}\right)c e^{-\alpha r}$:
1. **Overlap (Normalization) Integral:**
   $$\langle\Phi_\alpha|\Phi_\alpha\rangle = 4\pi c^2 \int_0^\infty r^2 e^{-2\alpha r}\,dr = 4\pi c^2 \frac{2!}{(2\alpha)^3} = \pi\frac{c^2}{\alpha^3}$$
2. **Kinetic Energy Integral:**
   $$\left\langle\Phi_\alpha\left|-\frac{1}{2}\nabla^2\right|\Phi_\alpha\right\rangle = -\frac{1}{2}(4\pi c^2)\int_0^\infty r^2\left(\alpha^2 - \frac{2\alpha}{r}\right)e^{-2\alpha r}\,dr = -2\pi c^2\left[\alpha^2\frac{2!}{(2\alpha)^3} - 2\alpha\frac{1!}{(2\alpha)^2}\right] = \pi\frac{c^2}{2\alpha}$$
3. **Coulomb Potential Energy Integral:**
   $$\left\langle\Phi_\alpha\left|-\frac{1}{r}\right|\Phi_\alpha\right\rangle = -4\pi c^2 \int_0^\infty r e^{-2\alpha r}\,dr = -4\pi c^2 \frac{1!}{(2\alpha)^2} = -\pi\frac{c^2}{\alpha^2}$$
4. **Total Variational Energy $E(\alpha)$ and Minimization (Slide 14):**
   Dividing by $\langle\Phi_\alpha|\Phi_\alpha\rangle = \pi\frac{c^2}{\alpha^3}$ cancels $\pi c^2$:
   $$\boxed{E(\alpha) = \frac{\pi\frac{c^2}{2\alpha} - \pi\frac{c^2}{\alpha^2}}{\pi\frac{c^2}{\alpha^3}} = \underbrace{\frac{1}{2}\alpha^2}_{T(\alpha)} \underbrace{- \alpha}_{V(\alpha)}}$$
   Setting $\frac{dE(\alpha)}{d\alpha} = \alpha - 1 = 0$ yields the optimal inverse decay length **$\alpha^* = 1\,a_0^{-1}$** and the exact ground-state energy:
   $$E(\alpha^* = 1) = \frac{1}{2}(1)^2 - 1 = -0.5\text{ Ha} = -13.6\text{ eV}$$


In [12]:
ws.render_section('concept_variational_hydrogen')

### Interactive Explorer: Variational Principle for the Hydrogen Atom

Vary the exponential decay parameter $\alpha \in [0.2, 2.2]\,a_0^{-1}$ for the trial wavefunction $\Phi_\alpha(r) = c e^{-\alpha r}$, and toggle between the energy components $T(\alpha), V(\alpha), E(\alpha)$ and the radial probability density $4\pi r^2|\Phi_\alpha(r)|^2$.


**Key Takeaway:** Squeezing the electron closer to the nucleus ($\alpha > 1$) makes the Coulomb potential $V(\alpha) = -\alpha$ more negative, but the kinetic energy $T(\alpha) = +\frac{1}{2}\alpha^2$ (Heisenberg uncertainty curvature) grows quadratically and wins! Conversely, delocalizing the electron ($\alpha < 1$) lowers $T(\alpha)$ but sacrifices too much Coulomb attraction. The exact balance occurs at $\alpha^* = 1\,a_0^{-1}$, where $E(1) = -0.50\text{ Ha}$ and the virial theorem $2T + V = 0$ is satisfied.


In [13]:
ws.render_section('q09_variational_principle_statement')

**Q9 · easy** · 💡

According to the Variational Principle (`05.QuantMech.pdf`, p. 12), if $E_0$ is the exact ground-state energy of a Hamiltonian $\hat{\mathcal{H}}$ and $\Phi$ is any normalized trial wavefunction ($\langle\Phi|\Phi\rangle = 1$) that is **not** identical to the exact ground state, what is always true of $E[\Phi] = \langle\Phi|\hat{\mathcal{H}}|\Phi\rangle$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Expand Phi in the complete orthonormal eigenstates of H: |Phi> = sum_n c_n |Psi_n> with E_n >= E_0.

</div>
</details>

- [ ] **(A)** E[Phi] is strictly greater than E_0 (E[Phi] > E_0), providing a rigorous upper bound to the true ground-state energy.
- [ ] **(B)** E[Phi] is strictly less than E_0 (E[Phi] < E_0).
- [ ] **(C)** E[Phi] can be either above or below E_0 depending on the sign of Phi.
- [ ] **(D)** E[Phi] = 0 for any non-eigenstate trial wavefunction.

In [14]:
ws.render_section('q10_atomic_units_values')

**Q10 · easy** · 💡

In Hartree atomic units (`05.QuantMech.pdf`, p. 15), $\hbar = m_e = e = \frac{1}{4\pi\varepsilon_0} = 1$. What are the approximate values of $1\text{ Bohr radius}$ ($a_0$) in Ångströms and $1\text{ Hartree}$ ($\text{Ha}$) in electron-volts ($\text{eV}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Convert 5.2918 x 10^-11 m to Angstroms (10^-10 m) and 4.3597 x 10^-18 J to eV (divide by 1.6022 x 10^-19 J/eV), or recall that the Hydrogen ground state is -0.5 Ha = -13.6 eV.

</div>
</details>

- [ ] **(A)** a_0 approx 0.529 A and 1 Ha approx 27.21 eV (so the Hydrogen ground state is E_0 = -0.5 Ha = -13.6 eV)
- [ ] **(B)** a_0 approx 1.000 A and 1 Ha approx 13.60 eV
- [ ] **(C)** a_0 approx 5.29 A and 1 Ha approx 1.00 eV
- [ ] **(D)** a_0 approx 0.053 A and 1 Ha approx 100.0 eV

In [15]:
ws.render_section('q11_hydrogen_overlap_integral')

**Q11 · medium** · 💡

Using the radial integral identity $\int_0^\infty r^n \exp(-ar)\,dr = \frac{n!}{a^{n+1}}$ from Slide 13, what is the overlap integral $\langle\Phi_\alpha|\Phi_\alpha\rangle = \int_0^\infty 4\pi r^2 \bigl(c e^{-\alpha r}\bigr)^2\,dr$ for the Hydrogen trial wavefunction $\Phi_\alpha(r) = c e^{-\alpha r}$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute n = 2 and a = 2*alpha into 4 * pi * c^2 * (n! / a^(n+1)).

</div>
</details>

- [ ] **(A)** pi * c^2 / alpha^3 (so normalization <Phi_alpha|Phi_alpha> = 1 requires c = sqrt(alpha^3 / pi))
- [ ] **(B)** 4 * pi * c^2 / alpha^2
- [ ] **(C)** pi * c^2 / (2 * alpha)
- [ ] **(D)** 2 * pi * c^2 / alpha^4

In [16]:
ws.render_section('q12_hydrogen_kinetic_potential_scaling')

**Q12 · medium** · 💡

In the Hydrogen variational derivation (Slide 14), after dividing by the normalization integral $\langle\Phi_\alpha|\Phi_\alpha\rangle = \pi\frac{c^2}{\alpha^3}$, how do the normalized kinetic energy $T(\alpha) = \frac{\langle\Phi_\alpha|-\frac{1}{2}\nabla^2|\Phi_\alpha\rangle}{\langle\Phi_\alpha|\Phi_\alpha\rangle}$ and potential energy $V(\alpha) = \frac{\langle\Phi_\alpha|-\frac{1}{r}|\Phi_\alpha\rangle}{\langle\Phi_\alpha|\Phi_\alpha\rangle}$ depend on $\alpha$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Divide pi*c^2/(2*alpha) and -pi*c^2/alpha^2 by pi*c^2/alpha^3 as shown on Slide 14.

</div>
</details>

- [ ] **(A)** T(alpha) = +(1/2)*alpha^2 and V(alpha) = -alpha, giving E(alpha) = (1/2)*alpha^2 - alpha
- [ ] **(B)** T(alpha) = +alpha and V(alpha) = -(1/2)*alpha^2, giving E(alpha) = alpha - (1/2)*alpha^2
- [ ] **(C)** T(alpha) = +1/(2*alpha^2) and V(alpha) = -1/alpha
- [ ] **(D)** T(alpha) = +(1/2)*alpha^3 and V(alpha) = -alpha^2

In [17]:
ws.render_section('q13_hydrogen_suboptimal_alpha_numeric')

**Q13 · easy** · 💡

Suppose you evaluate the Hydrogen variational energy $E(\alpha) = \frac{1}{2}\alpha^2 - \alpha$ (in Hartrees) at a compressed trial parameter $\alpha = 1.6\,a_0^{-1}$ instead of the optimal $\alpha^* = 1.0\,a_0^{-1}$. What is the resulting trial energy $E(1.6)$ (in $\text{Ha}$)? *(Include the correct algebraic sign.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Calculate 0.5 * (1.6)^2 - 1.6 = 1.28 - 1.60, and notice that it lies strictly above E_0 = -0.50 Ha!

</div>
</details>

_Numeric answer required._

In [18]:
ws.render_section('q14_hydrogenic_ion_z_scaling_numeric')

**Q14 · hard** · 💡

Consider a one-electron hydrogenic ion with nuclear charge $Z = 2$ ($\text{He}^+$), whose Hamiltonian in atomic units is $\hat{\mathcal{H}} = -\frac{1}{2}\nabla^2 - \frac{Z}{r} = -\frac{1}{2}\nabla^2 - \frac{2}{r}$. Using the same trial wavefunction $\Phi_\alpha(r) = c e^{-\alpha r}$, the potential energy integral is multiplied by $Z = 2$, giving $E(\alpha) = \frac{1}{2}\alpha^2 - Z\alpha = \frac{1}{2}\alpha^2 - 2\alpha$. Minimizing $E(\alpha)$ with respect to $\alpha$, what is the exact ground-state energy $E_0$ of $\text{He}^+$ (in $\text{Ha}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Set dE/dalpha = alpha - 2 = 0 to find alpha* = 2, then evaluate E(2) = 0.5 * (2)^2 - 2 * (2).

</div>
</details>

_Numeric answer required._

In [19]:
ws.render_section('sec_many_electron_and_hartree_fock')

## 3. The Many-Electron Problem, Hartree Method, and Hartree–Fock Exchange

For an atom, molecule, or solid with $n$ interacting electrons in the field of nuclear charge(s) $Z$, the many-electron time-independent Schrödinger equation in atomic units (`05.QuantMech.pdf`, p. 16) is:
$$\left[-\frac{1}{2}\sum_{i=1}^n \nabla_i^2 - \sum_{i=1}^n \frac{Z}{r_i} + \sum_{i=1}^{n-1}\sum_{j=i+1}^n \frac{1}{|\mathbf{r}_i - \mathbf{r}_j|}\right]\psi(\mathbf{r}_1, \dots, \mathbf{r}_n) = E\,\psi(\mathbf{r}_1, \dots, \mathbf{r}_n)$$

### The Curse of Dimensionality: Example of an $\text{Fe}$ Atom (Slide 16)
Because the two-electron Coulomb repulsion $\frac{1}{|\mathbf{r}_i - \mathbf{r}_j|}$ couples all electron coordinates together, $\psi(\mathbf{r}_1, \dots, \mathbf{r}_n)$ is a function of **$3n$ coupled spatial variables**!
- A single **iron ($\text{Fe}$) atom** has **$n = 26$ electrons** $\implies$ $\psi$ depends on $3 \times 26 = 78$ spatial variables.
- Even on a very coarse 3D grid with only $10$ grid points along each Cartesian axis ($10^3$ points per electron), storing $\psi(\mathbf{r}_1, \dots, \mathbf{r}_{26})$ requires **$10^{3 \times 26} = 10^{78}$ numbers**!
- At $4\text{ Bytes}$ per single-precision number ($4 \times 10^{78}\text{ Bytes}$), this dwarfs all digital data stored worldwide ($\sim 1\text{ zettabyte} = 10^{21}\text{ Bytes}$) by 57 orders of magnitude! Direct grid solution of the many-electron Schrödinger equation is physically impossible.

### 1. The Hartree Method (1928, Slides 17–19)
Douglas Hartree approximated the $n$-electron wavefunction as a simple **Hartree product** of independent single-particle orbitals:
$$\psi_{\text{Hartree}}(\mathbf{r}_1, \mathbf{r}_2, \dots, \mathbf{r}_n) = \varphi_1(\mathbf{r}_1)\,\varphi_2(\mathbf{r}_2)\cdots\varphi_n(\mathbf{r}_n)$$
Applying the variational principle yields a self-consistent equation where electron $i$ moves in the nuclear potential plus the mean-field **Hartree potential** of the other $j \ne i$ electron charge clouds:
$$\left[-\frac{1}{2}\nabla_i^2 + \sum_I V(\mathbf{R}_I - \mathbf{r}_i) + \underbrace{\sum_{j \ne i}\int |\varphi_j(\mathbf{r}_j)|^2 \frac{1}{|\mathbf{r}_i - \mathbf{r}_j|}\,d^3r_j}_{\text{Hartree potential}}\right]\varphi_i(\mathbf{r}_i) = \epsilon\,\varphi_i(\mathbf{r}_i)$$
**What is missing in the Hartree approximation (Slides 19–20)?**
1. **No Fermion Antisymmetry:** The Hartree product violates the Pauli Exclusion Principle.
2. **No Electron Correlation:** Electrons only feel the *average* smeared-out density of other electrons ("cars smeared out" rather than "cars avoiding each other" dynamically at short distances).

### 2. The Hartree–Fock (HF) Method & Slater Determinants (Slides 20–24)
Electrons are indistinguishable **fermions**: swapping any two electrons must flip the sign of the wavefunction:
$$\psi(\mathbf{r}_1, \dots, \mathbf{r}_j, \dots, \mathbf{r}_k, \dots, \mathbf{r}_n) = -\psi(\mathbf{r}_1, \dots, \mathbf{r}_k, \dots, \mathbf{r}_j, \dots, \mathbf{r}_n)$$
Fock and Slater enforced antisymmetry by replacing the Hartree product with a single **Slater Determinant** (Slide 21):
$$\psi_{\text{HF}}(\mathbf{r}_1, \mathbf{r}_2, \dots, \mathbf{r}_n) = \frac{1}{\sqrt{n!}} \begin{vmatrix} \varphi_1(\mathbf{r}_1) & \varphi_2(\mathbf{r}_1) & \cdots & \varphi_n(\mathbf{r}_1) \\ \varphi_1(\mathbf{r}_2) & \varphi_2(\mathbf{r}_2) & \cdots & \varphi_n(\mathbf{r}_2) \\ \vdots & \vdots & \ddots & \vdots \\ \varphi_1(\mathbf{r}_n) & \varphi_2(\mathbf{r}_n) & \cdots & \varphi_n(\mathbf{r}_n) \end{vmatrix}$$
- Swapping two electron coordinates $\mathbf{r}_j \leftrightarrow \mathbf{r}_k$ swaps two rows of the determinant, automatically changing its sign! If two electrons occupy the same spin-orbital ($\varphi_\mu = \varphi_\nu$), two columns are identical and $\psi_{\text{HF}} \equiv 0$ (**Pauli Exclusion Principle**).
- Applying the variational principle to $\psi_{\text{HF}}$ yields the **Hartree–Fock equations** (Slides 22–23):
  $$\left[-\frac{1}{2}\nabla_i^2 + \sum_I V(\mathbf{R}_I - \mathbf{r}_i)\right]\varphi_\lambda(\mathbf{r}_i) + \left[\sum_\mu \int \varphi_\mu^*(\mathbf{r}_j)\frac{1}{|\mathbf{r}_i - \mathbf{r}_j|}\varphi_\mu(\mathbf{r}_j)\,d^3r_j\right]\varphi_\lambda(\mathbf{r}_i) - \underbrace{\sum_\mu \left[\int \varphi_\mu^*(\mathbf{r}_j)\frac{1}{|\mathbf{r}_i - \mathbf{r}_j|}\varphi_\lambda(\mathbf{r}_j)\,d^3r_j\right]\varphi_\mu(\mathbf{r}_i)}_{\text{Non-local Exchange Term } \int V_\lambda^x(\mathbf{r}_i, \mathbf{r}_j)\varphi_\lambda(\mathbf{r}_j)\,d^3r_j} = \epsilon\,\varphi_\lambda(\mathbf{r}_i)$$
- **Successes of HF:** Exact Pauli exchange; **completely self-interaction free** (for $\mu = \lambda$, the $\mu=\lambda$ exchange integral exactly cancels the spurious $\mu=\lambda$ self-Hartree integral!); good atomic properties; foundation for correlated quantum chemistry.
- **Limitations of HF & Correlation Energy (Slide 24):** By definition, any effect beyond a single Slater determinant is **electron correlation**:
  $$\boxed{E_{\text{corr}} = E_{\text{exact}} - E_{\text{HF}} \le 0}$$
  Although $|E_{\text{corr}}|$ is typically $< 1\%$ of the total energy $E_{\text{exact}}$ (e.g., for $\text{N}_2$, $|E_{\text{corr}}| = 14.9\text{ eV} < 1\%$ of $E_{\text{exact}}$), it makes a **huge** contribution to chemical bonding! For $\text{N}_2 \to \text{N} + \text{N}$, Hartree–Fock predicts a binding energy of only $D_e(\text{HF}) = 5.1\text{ eV}$ vs. $D_e(\text{exp}) = 9.9\text{ eV}$ (severe underbinding!).

### 3. Beyond Hartree–Fock: Configuration Interaction & Thomas–Fermi (Slides 25–26)
- **Configuration Interaction (CI):** Expands the many-body wavefunction as a linear combination of the ground-state HF determinant $\Psi_0$ plus excited Slater determinants $\Psi_1, \Psi_2, \dots$ formed by promoting electrons from occupied orbitals $i$ to unoccupied ("virtual") orbitals $K+1$:
  $$\Psi_{\text{CI}} = c_0 \Psi_0 + c_1 \Psi_1 + c_2 \Psi_2 + \dots$$
  Full CI approaches $E_{\text{exact}}$ as the basis set grows, but scales combinatorially with $n$.
- **Thomas–Fermi Approach (1927, Slide 26):** Proposed replacing the $3n$-variable wavefunction $\Psi(\mathbf{r}_1, \dots, \mathbf{r}_n)$ with the **3-variable electron density $\rho(\mathbf{r})$** by modeling the kinetic energy at each point as a *locally homogeneous electron gas* ($T_{\text{TF}}[\rho] \propto \int \rho(\mathbf{r})^{5/3}\,d^3r$). While Thomas–Fermi failed to bind molecules (due to its crude orbital-free kinetic energy), it inspired modern **Density Functional Theory**!


In [20]:
ws.render_section('q15_fe_atom_grid_storage_numeric')

**Q15 · easy** · 💡

As shown in the $\text{Fe}$ atom example on Slide 16 (`05.QuantMech.pdf`), an iron atom has $n = 26$ electrons in 3D space. If we discretize each spatial coordinate $(x_i, y_i, z_i)$ on a coarse grid of just $10$ points per direction, the number of grid values required to store the full many-body wavefunction $\psi(\mathbf{r}_1, \dots, \mathbf{r}_{26})$ is $10^P$. What is the exponent $P$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each of the 26 electrons has 3 spatial coordinates, giving 3 * 26 variables, each with 10 grid points.

</div>
</details>

_Numeric answer required._

In [21]:
ws.render_section('q16_hartree_method_deficiencies')

**Q16 · easy** · 💡

According to Slides 17–20 of `05.QuantMech.pdf`, what are the two fundamental physical properties missing from the 1928 Hartree product approximation $\psi(\mathbf{r}_1, \dots, \mathbf{r}_n) = \varphi_1(\mathbf{r}_1)\varphi_2(\mathbf{r}_2)\cdots\varphi_n(\mathbf{r}_n)$? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the bullet points under 'What is missing in the Hartree approximation' on Slide 20.

</div>
</details>

- [ ] **(A)** The wavefunction is not antisymmetric under the exchange of two identical fermions (violating the Pauli Exclusion Principle).
- [ ] **(B)** It does not include dynamic electron correlation (electrons only interact with the smeared-out average density of other electrons and can get arbitrarily close).
- [ ] **(C)** It omits the attraction between electrons and atomic nuclei.
- [ ] **(D)** It omits the single-particle kinetic energy operator -1/2 nabla_i^2.

In [22]:
ws.render_section('q17_slater_determinant_antisymmetry')

**Q17 · easy** · 💡

How does representing an $n$-electron wavefunction as a normalized **Slater determinant** (Slide 21) automatically enforce both fermion antisymmetry and the Pauli Exclusion Principle?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall two basic linear algebra properties of determinants: what happens when two rows are swapped, and what happens when two columns are identical?

</div>
</details>

- [ ] **(A)** Exchanging two electrons swaps two rows of the determinant, which flips the sign of psi; placing two electrons in the same quantum state makes two columns identical, causing the determinant to vanish (psi = 0).
- [ ] **(B)** The determinant of a matrix is always positive definite, preventing negative probabilities.
- [ ] **(C)** The prefactor 1/sqrt(n!) sets the Coulomb repulsion between all electron pairs to zero.
- [ ] **(D)** Swapping two rows of a determinant leaves its sign unchanged.

In [23]:
ws.render_section('q18_slater_determinant_normalization_numeric')

**Q18 · easy** · 💡

In the Slater determinant formula $\psi(\mathbf{r}_1, \dots, \mathbf{r}_n) = \frac{1}{\sqrt{n!}}\det[\varphi_j(\mathbf{r}_i)]$ (Slide 21) built from $n$ orthonormal orbitals, the determinant expands into $n!$ permutation terms. For a $4$-electron Be atom ($n = 4$), how many product terms are in the expanded determinant ($n!$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute 4! = 4 * 3 * 2 * 1.

</div>
</details>

_Numeric answer required._

In [24]:
ws.render_section('q19_hf_nonlocal_exchange_term')

**Q19 · medium** · 💡

Why is the Hartree–Fock exchange term $-\sum_\mu \left[\int \varphi_\mu^*(\mathbf{r}_j)\frac{1}{|\mathbf{r}_i - \mathbf{r}_j|}\varphi_\lambda(\mathbf{r}_j)\,d^3r_j\right]\varphi_\mu(\mathbf{r}_i)$ called a **non-local potential** on Slide 23 (`05.QuantMech.pdf`)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare V_H(r_i) * phi_lambda(r_i) (which only depends on phi_lambda at point r_i) against the exchange integral (where phi_lambda(r_j) is inside the integral over all r_j!).

</div>
</details>

- [ ] **(A)** Because the orbital phi_lambda(r_j) appears inside the integral over r_j (of the form integral V_lambda^x(r_i, r_j) phi_lambda(r_j) d^3r_j) and cannot be written as a simple multiplicative function V^x(r_i) * phi_lambda(r_i) at a single point r_i.
- [ ] **(B)** Because it acts only on electrons located infinitely far outside the crystal.
- [ ] **(C)** Because the Coulomb potential 1/|r_i - r_j| is independent of distance.
- [ ] **(D)** Because it violates conservation of charge.

In [25]:
ws.render_section('q20_hf_self_interaction_cancellation')

**Q20 · medium** · 💡

In the Hartree–Fock equation on Slide 22, the Hartree sum $\sum_\mu$ includes the $\mu = \lambda$ term (where an electron in orbital $\varphi_\lambda$ repels its own charge cloud). Why is Hartree–Fock nevertheless **completely free of self-interaction error** (Slide 24)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the mu = lambda term in the Hartree sum against the mu = lambda term in the subtracted Exchange sum on Slide 22.

</div>
</details>

- [ ] **(A)** For mu = lambda, the exchange term integral is identical to the Hartree self-repulsion integral with the opposite sign, so they cancel out exactly!
- [ ] **(B)** In Hartree-Fock, the Coulomb repulsion 1/|r_i - r_j| is set to zero for all orbitals.
- [ ] **(C)** Self-interaction only occurs for nuclei, not for electrons.
- [ ] **(D)** The kinetic energy cancels the Hartree self-interaction.

In [26]:
ws.render_section('q21_correlation_energy_and_n2_binding')

**Q21 · medium** · 💡

On Slide 24 (`05.QuantMech.pdf`), the correlation energy is defined as $E_{\text{corr}} = E_{\text{exact}} - E_{\text{HF}}$. For the $\text{N}_2$ molecule, $|E_{\text{corr}}| = 14.9\text{ eV}$, which is $< 1\%$ of the total energy $E_{\text{exact}}$. Why is neglecting $E_{\text{corr}}$ in Hartree–Fock still unacceptable for chemistry and materials science?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the HF binding energy of N_2 (5.1 eV) against the experimental binding energy (9.9 eV) on Slide 24.

</div>
</details>

- [ ] **(A)** Chemical reactions and cohesive binding depend on relative energy differences rather than total energies; for N_2 -> N + N, correlation accounts for nearly half the bond dissociation energy (D_e(HF) = 5.1 eV vs. D_e(exp) = 9.9 eV).
- [ ] **(B)** Because E_HF is always lower than E_exact, violating the variational principle.
- [ ] **(C)** Because Hartree-Fock predicts N_2 to have a binding energy of 1000 eV.
- [ ] **(D)** Because correlation energy only affects core 1s electrons and never changes chemical bond energies.

In [27]:
ws.render_section('q22_n2_correlation_binding_error_numeric')

**Q22 · easy** · 💡

From Slide 24 (`05.QuantMech.pdf`), the experimental dissociation (binding) energy of $\text{N}_2 \to \text{N} + \text{N}$ is $D_e(\text{exp}) = 9.9\text{ eV}$, whereas Hartree–Fock predicts $D_e(\text{Hartree–Fock}) = 5.1\text{ eV}$. By how many $\text{eV}$ does Hartree–Fock underestimate the binding energy of $\text{N}_2$ due to missing electron correlation?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute D_e(exp) - D_e(Hartree-Fock) = 9.9 - 5.1.

</div>
</details>

_Numeric answer required._

In [28]:
ws.render_section('q23_configuration_interaction_and_thomas_fermi')

**Q23 · medium** · 💡

How do the **Configuration Interaction (CI)** method (Slide 25) and the **Thomas–Fermi** approach (Slide 26) take opposite philosophical routes to go beyond Hartree–Fock?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare expanding Psi_CI = c_0 Psi_0 + c_1 Psi_1 + ... in excited Slater determinants vs. replacing Psi(r_1,...,r_n) with the 3-variable density rho(r).

</div>
</details>

- [ ] **(A)** CI systematically improves the 3n-variable wavefunction by mixing in excited Slater determinants (Psi_CI = sum c_k Psi_k), whereas Thomas-Fermi replaces the 3n-variable wavefunction altogether with the 3-variable electron density rho(r).
- [ ] **(B)** CI replaces the wavefunction with the 3D density rho(r), whereas Thomas-Fermi mixes excited Slater determinants.
- [ ] **(C)** Both CI and Thomas-Fermi use a single unantisymmetrized Hartree product.
- [ ] **(D)** Thomas-Fermi is an exact correlated wavefunction method that scales as O(N!).

In [29]:
ws.render_section('sec_dft_hohenberg_kohn_kohn_sham')

## 4. Density Functional Theory: Hohenberg–Kohn Theorems, Kohn–Sham Equations & Jacob's Ladder

**Density Functional Theory (DFT)** — for which Walter Kohn was awarded the 1998 Nobel Prize in Chemistry (`05.QuantMech.pdf`, pp. 27–36) — places the density-based approach on an **exact** mathematical footing, reducing the $3N$-variable problem to a **3-variable electron density $\rho(\mathbf{r})$**!

### 1st Hohenberg–Kohn Theorem (1964, Slides 28–29)
- Normally, the external nuclear potential $V_{\text{ext}}(\mathbf{r})$ and electron count $N_{\text{el}}$ define the Hamiltonian $\hat{\mathcal{H}}$, which determines the many-body ground state $\Psi_0(\mathbf{r}_1, \dots, \mathbf{r}_N)$ and, by integrating over $N-1$ coordinates, the ground-state density $\rho_0(\mathbf{r})$.
- **Hohenberg and Kohn proved that this arrow is invertible:** the ground-state electron density $\rho_0(\mathbf{r})$ **uniquely determines** the external potential $V_{\text{ext}}(\mathbf{r})$ (up to an additive constant), and therefore uniquely determines $\Psi_0$ and **all** ground-state properties!
- Consequently, the ground-state total energy is a unique functional of the density:
  $$E_{\text{tot}}^0[\rho_0] = \underbrace{E_{\text{kin}}[\rho_0] + V_{\text{el-el}}[\rho_0]}_{\text{Universal functional } F[\rho_0]} + \int V_{\text{ext}}(\mathbf{r})\rho_0(\mathbf{r})\,d^3r$$
  where $F[\rho] = E_{\text{kin}}[\rho] + V_{\text{el-el}}[\rho]$ is a **universal functional** (its mathematical form is identical for any atom, molecule, or solid, independent of $V_{\text{ext}}$), though its exact analytical formula is unknown.

### 2nd Hohenberg–Kohn Theorem (Slide 30)
- The ground-state energy and density can be obtained **variationally**: for any trial 3D electron density $\rho(\mathbf{r})$ integrating to $N$ electrons,
  $$\boxed{E[\rho(\mathbf{r})] = F[\rho(\mathbf{r})] + \int V_{\text{ext}}(\mathbf{r})\rho(\mathbf{r})\,d^3r \ge E_0}$$
  and the density $\rho(\mathbf{r})$ that minimizes $E[\rho]$ is the **exact ground-state density $\rho_0(\mathbf{r})$**!

### The Kohn–Sham Equations (1965, Slides 31–33)
Why couldn't we just minimize $E[\rho]$ directly? Because orbital-free approximations (like Thomas–Fermi) fail badly for the kinetic energy $E_{\text{kin}}[\rho]$.
**Kohn and Sham's brilliant breakthrough (Slide 31):** map the interacting electron system onto an **auxiliary system of non-interacting electrons** $\{\varphi_i(\mathbf{r})\}_{i=1}^N$ constructed to have the **exact same ground-state density** $\rho(\mathbf{r}) = \sum_{i=1}^N |\varphi_i(\mathbf{r})|^2$!
- Why? Because the kinetic energy of non-interacting orbitals is known exactly:
  $$T_S[\rho(\mathbf{r})] = -\frac{1}{2}\sum_{i=1}^N \int \varphi_i^*(\mathbf{r})\nabla^2 \varphi_i(\mathbf{r})\,d^3r$$
- The universal functional is then partitioned into the known non-interacting kinetic energy $T_S[\rho]$, the known classical Hartree electrostatic energy $E_H[\rho]$, and the **exchange-correlation functional $E_{\text{xc}}[\rho]$** (which absorbs all remaining quantum exchange, correlation, and the small kinetic correlation difference $E_{\text{kin}} - T_S$):
  $$F[\rho(\mathbf{r})] = T_S[\rho(\mathbf{r})] + \underbrace{\frac{1}{2}\iint \frac{\rho(\mathbf{r})\rho(\mathbf{r}')}{|\mathbf{r}-\mathbf{r}'|}\,d^3r\,d^3r'}_{E_H[\rho(\mathbf{r})]} + E_{\text{xc}}[\rho(\mathbf{r})]$$
- Applying the Euler–Lagrange variational minimization $\frac{\delta}{\delta\rho}\left\{E[\rho] - \mu\left(\int\rho(\mathbf{r})\,d^3r - N\right)\right\} = 0$ yields the single-particle **Kohn–Sham Equations** (Slide 32):
  $$\boxed{\underbrace{\left[-\frac{1}{2}\nabla^2 + V_H(\mathbf{r}) + V_{\text{xc}}(\mathbf{r}) + V_{\text{ext}}(\mathbf{r})\right]}_{\hat{\mathcal{H}}_{\text{KS}}}\varphi_i(\mathbf{r}) = \epsilon_i\,\varphi_i(\mathbf{r})}$$
  where $V_H(\mathbf{r}) = \int \frac{\rho(\mathbf{r}')}{|\mathbf{r}-\mathbf{r}'|}\,d^3r'$ and the **local exchange-correlation potential** is the functional derivative $V_{\text{xc}}(\mathbf{r}) = \frac{\delta E_{\text{xc}}[\rho]}{\delta\rho(\mathbf{r})}$.
- **Total Energy Double-Counting Correction (Slide 33):** Notice that the sum of Kohn–Sham eigenvalues $\sum_{i=1}^N \epsilon_i$ double-counts the Hartree energy and uses $\int V_{\text{xc}}(\mathbf{r})\rho(\mathbf{r})\,d^3r$ instead of $E_{\text{xc}}[\rho]$:
  $$E[\{\varphi_i\}] = \sum_{i=1}^N \epsilon_i - E_H[\rho(\mathbf{r})] + E_{\text{xc}}[\rho(\mathbf{r})] - \int \frac{\delta E_{\text{xc}}[\rho]}{\delta\rho(\mathbf{r})}\rho(\mathbf{r})\,d^3r$$

### Exchange–Correlation Functionals: Jacob's Ladder (Slides 34–36, 52)
Since Kohn–Sham DFT is exact in principle if $E_{\text{xc}}[\rho]$ were known, all practical approximations lie in modeling $E_{\text{xc}}[\rho]$:
1. **Local Density Approximation (LDA, Slides 34–36):** Assumes $E_{\text{xc}}^{\text{LDA}}[\rho] = \int \rho(\mathbf{r})\varepsilon_{\text{xc}}^{\text{HEG}}(\rho(\mathbf{r}))\,d^3r$, where $\varepsilon_{\text{xc}}^{\text{HEG}}(\rho)$ is the exact exchange-correlation energy per electron of a **uniform (homogeneous) electron gas**, parameterized from **Ceperley and Alder's (1980) Quantum Monte Carlo** simulations (which also mapped the low-density Wigner crystal transition at $r_s \approx 100$).
   - *Performance:* Surprisingly accurate for solid phase stability (e.g. correctly predicts the energetic order of all **11 high-pressure crystal phases of Silicon**, Slide 35!), but systematically **overbinds** solids (**cohesive energies 5–20% too large**, **lattice constants 1–3% too small**, bulk moduli 5–20% too stiff).
2. **Generalized Gradient Approximation (GGA: PW91, PBE, Slide 36):** Incorporates both the local density $\rho(\mathbf{r})$ and its spatial gradient $|\nabla\rho(\mathbf{r})|$ while preserving exact analytic scaling laws. Improves cohesive energies and is crucial for magnetic transition metals (e.g. predicting BCC ferromagnetic Fe as the ground state).
3. **Meta-GGA (TPSS, SCAN):** Includes the Laplacian $\nabla^2\rho$ or orbital kinetic-energy density $\tau(\mathbf{r}) = \frac{1}{2}\sum_i |\nabla\varphi_i|^2$ (curvature information).
4. **Hybrid Functionals (B3LYP, HSE, Slide 36):** Mix a fraction (typically 20–25%) of **non-local Hartree–Fock exact exchange** with GGA/Meta-GGA exchange-correlation, partially curing self-interaction error and dramatically improving **semiconductor band gaps**.


In [30]:
ws.render_section('concept_kohn_sham_scf')

### Interactive Explorer: Kohn–Sham Self-Consistent Field (SCF) Loop & Functional Benchmarks

Because the Kohn–Sham effective potential $V_{\text{KS}}[\rho] = V_{\text{ext}} + V_H[\rho] + V_{\text{xc}}[\rho]$ depends on the electron density $\rho(\mathbf{r}) = \sum_i |\varphi_i(\mathbf{r})|^2$ produced by its own eigenfunctions, the Kohn–Sham equations must be solved iteratively until self-consistency! Vary the density mixing parameter $\beta$ and select an exchange-correlation functional along Jacob's Ladder.


**Key Takeaway:** By the 2nd Hohenberg–Kohn theorem, $\left.\frac{\delta E[\rho]}{\delta\rho}\right|_{\rho_0} = \mu$, so a first-order density error $\|\delta\rho\|$ produces a **second-order** variational energy error $|E[\rho] - E_0| \propto \|\delta\rho\|^2$. Notice how choosing $\beta$ too large ($\beta \gtrsim 0.9$) triggers *charge sloshing* divergence!


In [31]:
ws.render_section('q24_first_hohenberg_kohn_theorem')

**Q24 · easy** · 💡

What is the central statement of the **1st Hohenberg–Kohn Theorem** (`05.QuantMech.pdf`, pp. 28–29) that makes Density Functional Theory possible?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the reverse arrow V_ext(r) <= rho_0(r) in the box on Slide 28.

</div>
</details>

- [ ] **(A)** The ground-state electron density rho_0(r) (a function of only 3 spatial variables) uniquely determines the external potential V_ext(r), and therefore uniquely determines the many-body ground-state wavefunction Psi_0 and all system properties.
- [ ] **(B)** Any excited-state wavefunction is an explicit polynomial of the nuclear positions.
- [ ] **(C)** The electron-electron Coulomb repulsion vanishes in all crystalline solids.
- [ ] **(D)** The kinetic energy of an interacting electron gas is identically zero.

In [32]:
ws.render_section('q25_universal_functional_definition')

**Q25 · medium** · 💡

In the Hohenberg–Kohn energy decomposition $E_{\text{tot}}^0[\rho] = F[\rho] + \int V_{\text{ext}}(\mathbf{r})\rho(\mathbf{r})\,d^3r$ (Slide 29), what does $F[\rho]$ contain and why is it called the **universal functional**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check Slide 29 for the definition F[rho^0] = E_kin[rho^0] + V_{el-el}[rho^0].

</div>
</details>

- [ ] **(A)** F[rho] = E_kin[rho] + V_{el-el}[rho] contains the electronic kinetic energy and electron-electron interaction; it is 'universal' because its functional form is exactly the same for any atom, molecule, or solid (independent of the external nuclear potential V_ext).
- [ ] **(B)** F[rho] contains only the nuclear-nuclear repulsion and changes functional form for every element in the periodic table.
- [ ] **(C)** F[rho] is called universal because its simple closed-form algebraic formula was derived by Thomas and Fermi in 1927.
- [ ] **(D)** F[rho] = int V_ext(r) rho(r) d^3r.

In [33]:
ws.render_section('q26_second_hohenberg_kohn_theorem')

**Q26 · easy** · 💡

What does the **2nd Hohenberg–Kohn Theorem** (Slide 30) guarantee about the total energy functional $E[\rho(\mathbf{r})] = F[\rho(\mathbf{r})] + \int V_{\text{ext}}(\mathbf{r})\rho(\mathbf{r})\,d^3r$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the two bullet points and the inequality on Slide 30.

</div>
</details>

- [ ] **(A)** E[rho(r)] >= E_0 for any valid trial density rho(r), and the density that minimizes E[rho(r)] is the exact ground-state density rho_0(r).
- [ ] **(B)** E[rho(r)] <= E_0 for all trial densities.
- [ ] **(C)** Every trial density yields the exact same total energy E_0.
- [ ] **(D)** The 2nd Hohenberg-Kohn theorem applies only to a single isolated hydrogen atom.

In [34]:
ws.render_section('q27_kohn_sham_mapping_motivation')

**Q27 · medium** · 💡

In the Kohn–Sham formulation of DFT (Slides 31–33), why do we re-introduce $N$ single-particle orbitals $\varphi_i(\mathbf{r})$ ($\rho(\mathbf{r}) = \sum_{i=1}^N |\varphi_i(\mathbf{r})|^2$) describing a fictitious **non-interacting** reference system, instead of minimizing $E[\rho]$ directly using $\rho(\mathbf{r})$ alone?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the first bullet point on Slide 31 ('Why? The kinetic energy of the non-interacting system is well defined.').

</div>
</details>

- [ ] **(A)** Because the kinetic energy functional of an interacting density E_kin[rho] is unknown and poorly approximated by orbital-free formulas, whereas the non-interacting orbital kinetic energy T_S = -1/2 sum_i <phi_i|nabla^2|phi_i> is exact and captures almost all of the true kinetic energy!
- [ ] **(B)** Because the Kohn-Sham orbitals eliminate the need to compute the Hartree electrostatic potential V_H(r).
- [ ] **(C)** Because the Kohn-Sham orbitals make the exchange-correlation functional E_xc[rho] identically zero.
- [ ] **(D)** Because a 3N-variable Slater determinant is faster to store on a grid than a 3D density.

In [35]:
ws.render_section('q28_kohn_sham_effective_potential')

**Q28 · medium** · 💡

In the Kohn–Sham eigenvalue equation $\left[-\frac{1}{2}\nabla^2 + V_{\text{eff}}(\mathbf{r})\right]\varphi_i(\mathbf{r}) = \epsilon_i\varphi_i(\mathbf{r})$ (Slide 32), what are the three terms that make up the local Kohn–Sham effective potential $V_{\text{eff}}(\mathbf{r})$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Inspect the boxed equation for H_KS on Slide 32.

</div>
</details>

- [ ] **(A)** V_eff(r) = V_H(r) + V_xc(r) + V_ext(r), where V_H is the classical Hartree potential, V_xc = delta E_xc / delta rho(r) is the local exchange-correlation potential, and V_ext is the external nuclear potential.
- [ ] **(B)** V_eff(r) = T_S(r) + E_corr(r) - V_ext(r).
- [ ] **(C)** V_eff(r) contains only the external nuclear attraction V_ext(r) because Kohn-Sham electrons do not feel V_H or V_xc.
- [ ] **(D)** V_eff(r) is the non-local Hartree-Fock integral operator V_lambda^x(r, r') with zero correlation.

In [36]:
ws.render_section('q29_kohn_sham_total_energy_double_counting')

**Q29 · hard** · 💡

According to the summary equation on Slide 33 (`05.QuantMech.pdf`), why is the Kohn–Sham total ground-state energy **not** simply equal to the sum of the occupied Kohn–Sham eigenvalues $\sum_{i=1}^N \epsilon_i$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the bottom line of Slide 33: E = sum_i eps_i - E_H[rho] + E_xc[rho] - int (delta E_xc / delta rho) * rho d^3r.

</div>
</details>

- [ ] **(A)** Because summing eps_i = <phi_i | -1/2 nabla^2 + V_H + V_xc + V_ext | phi_i> double-counts the Hartree energy (giving 2*E_H instead of E_H) and integrates V_xc(r)*rho(r) instead of E_xc[rho], requiring the correction E = sum_i eps_i - E_H[rho] + E_xc[rho] - int V_xc(r)*rho(r) d^3r.
- [ ] **(B)** Because the Kohn-Sham eigenvalues eps_i do not include the kinetic energy T_S.
- [ ] **(C)** Because the external potential V_ext(r) is double-counted in sum_i eps_i.
- [ ] **(D)** Because sum_i eps_i is always identically zero in a periodic crystal.

In [37]:
ws.render_section('q30_lda_ceperley_alder_origin')

**Q30 · easy** · 💡

In the **Local Density Approximation (LDA)** (`05.QuantMech.pdf`, pp. 34, 36), how was the exchange-correlation energy density of the uniform (homogeneous) electron gas determined?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the citation on Slide 34 and the first bullet point on Slide 36.

</div>
</details>

- [ ] **(A)** From Ceperley and Alder's (1980) stochastic Quantum Monte Carlo (QMC) simulations of the uniform electron gas across electron densities.
- [ ] **(B)** By fitting empirical parameters to the experimental lattice constant of diamond silicon.
- [ ] **(C)** By setting the correlation energy equal to zero and using only the Thomas-Fermi kinetic energy.
- [ ] **(D)** From classical molecular dynamics with a Lennard-Jones 12-6 potential.

In [38]:
ws.render_section('q31_jacobs_ladder_functionals')

**Q31 · medium** · 💡

On Slide 36 (`05.QuantMech.pdf`), four classes of exchange-correlation functionals are presented in order of increasing physical ingredients: **LDA**, **GGA (PW91, PBE)**, **Meta-GGA (TPSS)**, and **Hybrid Functionals (B3LYP, HSE)**. Which option correctly matches each rung to the new physical ingredient it introduces?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review the bullets under LDA, GGA, Meta-GGA, and Hybrid density functionals on Slide 36.

</div>
</details>

- [ ] **(A)** LDA uses only local density rho(r); GGA adds density gradients |nabla rho|; Meta-GGA adds density curvature / kinetic-energy density; Hybrids mix in a fraction of non-local Hartree-Fock exact exchange.
- [ ] **(B)** LDA includes Hartree-Fock exchange; GGA removes gradients; Meta-GGA uses only uniform gas density.
- [ ] **(C)** All four functionals depend solely on the local density rho(r) at a single point.
- [ ] **(D)** Hybrid functionals replace the Kohn-Sham equations with empirical pair potentials.

In [39]:
ws.render_section('q32_lda_vs_gga_systematic_trends')

**Q32 · medium** · 💡

According to the DFT summary on Slide 52 (`05.QuantMech.pdf`), which of the following statements accurately describe the systematic errors and strengths of **LDA** and **GGA** for solids? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the summary bullets for LDA and GGA on Slide 52.

</div>
</details>

- [ ] **(A)** LDA overbinds solids: its cohesive energies are 5-20% too strongly bound, and its equilibrium lattice constants are 1-3% too small.
- [ ] **(B)** LDA bulk moduli have 5-20% errors (with the largest errors occurring for late transition metals).
- [ ] **(C)** GGA improves cohesive energies over LDA, is often (though not always) better for lattice parameters, and is especially important for magnetic systems.
- [ ] **(D)** Both LDA and GGA predict semiconductor band gaps that are 50% too large compared to experiment.

In [40]:
ws.render_section('q33_silicon_high_pressure_phases')

**Q33 · easy** · 💡

As illustrated on Slide 35 (`05.QuantMech.pdf`), when Silicon is compressed from ambient pressure up to $79\text{ GPa}$, it undergoes a sequence of phase transitions across 11 crystal structures: $\text{Si(I) diamond } (Z=4) \to \text{Si(II) }\beta\text{-tin } (Z=6) \to \text{Imma} \to \text{simple hexagonal } (Z=8) \to \text{orthorhombic } (Z=10) \to \text{hcp } (Z=12) \to \text{fcc } (Z=12)$. What is the physical trend in coordination number $Z$ and electronic character under compression, and how well does DFT-LDA perform?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Inspect the coordination numbers Z = 4 -> 6 -> 8 -> 10 -> 12 and the caption at the bottom of Slide 35.

</div>
</details>

- [ ] **(A)** Coordination number steadily increases from Z = 4 (open covalent diamond) to Z = 12 (close-packed metallic hcp/fcc) accompanied by an insulator-to-metal transition, and DFT-LDA correctly predicts the energetic order of all these phases!
- [ ] **(B)** Coordination number decreases from Z = 12 at ambient pressure to Z = 2 at 79 GPa, and LDA fails to predict any phase transition.
- [ ] **(C)** Silicon remains in the 4-coordinated diamond structure at all pressures up to 100 GPa.
- [ ] **(D)** High-pressure silicon phases are wider-band-gap insulators than ambient diamond silicon.

In [41]:
ws.render_section('sec_pseudopotentials_and_planewaves')

## 5. DFT in Practice: Pseudopotentials, Plane Waves & Reciprocal Space

As outlined on Slide 37 (`05.QuantMech.pdf`), running a practical solid-state DFT calculation involves four pillars:
1. **Remove tightly bound core electrons** via the **pseudopotential approach**.
2. **Expand valence orbitals in a basis set** (localized **Gaussians/atomic orbitals** for isolated molecules; periodic **plane waves** for crystalline solids).
3. **Evaluate total energy using Fast Fourier Transforms (FFTs)** between real and reciprocal space (computing **kinetic and Hartree energies in reciprocal space**, and **exchange-correlation and external potentials in real space**) plus **Brillouin-zone $\mathbf{k}$-point integrations**.
4. **Minimize energy / iterate charge density to self-consistency.**

### Why Pseudopotentials Are Essential (Slides 38–40)
Consider **Aluminum ($\text{Al}, Z = 13$, Slide 38)**:
- Its 10 **core electrons** ($1s^2$ at $-1512\text{ eV}$, $2s^2$ at $-108\text{ eV}$, $2p^6$ at $-69.8\text{ eV}$) are tightly bound near the nucleus and **do not participate in chemical bonding** ("frozen core").
- Only the 3 **valence electrons** ($3s^2$ at $-7.8\text{ eV}$, $3p^1$ at $-2.7\text{ eV}$) form metallic/chemical bonds, allowing us to replace $\text{Al } (Z=13)$ with **Pseudo-$\text{Al } (Z_{\text{val}} = 3)$**!
- Furthermore, inside the core region ($r < r_c$, Slide 39), the true all-electron valence wavefunction $\Psi_{\text{AE}}(r)$ **oscillates rapidly** (it must have radial nodes to remain orthogonal to the core $1s, 2s, 2p$ states!), and the bare nuclear Coulomb potential $V(r) \sim -Z/r$ diverges at $r \to 0$. Resolving those rapid short-wavelength core wiggles in a plane-wave basis would require an astronomical plane-wave cutoff energy $E_{\text{cut}}$!
- **Pseudopotential Construction (Slide 39):** Inside a chosen cutoff radius $r < r_c$, we replace the diverging $-Z/r$ potential with a smooth $V_{\text{pseudo}}(r)$ and replace the oscillating $\Psi_{\text{AE}}(r)$ with a **smooth, nodeless pseudo-wavefunction $\Psi_{\text{pseudo}}(r)$**, subject to the constraint that **outside the cutoff region ($r \ge r_c$), $\Psi_{\text{pseudo}}(r) \equiv \Psi_{\text{AE}}(r)$, $V_{\text{pseudo}}(r) \equiv V_{\text{AE}}(r)$, and the eigenvalues $\epsilon_i$ are identical**!

### Non-Local Pseudopotentials (Slide 40)
Why must accurate pseudopotentials be **non-local** (i.e., angular-momentum dependent, having separate potentials $V_s(r), V_p(r), V_d(r)$ for different $l$ channels)?
- As explained on Slide 40, **valence states with different angular momenta $l$ scatter differently from the core**.
- Specifically, valence states that have a **core shell of the same angular momentum $l$ below them** feel a strong effective Pauli repulsion (orthogonality constraint to the inner core shell of the same $l$), whereas the first shell of a given $l$ (such as $2p$ in Carbon/Oxygen or $3d$ in $3d$ transition metals, which have **no** core $p$ or $d$ shell below them!) feels no radial orthogonality repulsion.


In [42]:
ws.render_section('concept_pseudopotential_smoothing')

### Interactive Explorer: Pseudopotential Smoothing & Plane-Wave Basis Cutoff

Adjust the pseudopotential core cutoff radius $r_c$ and switch between the **Real-Space Wavefunction/Potential** view (matching Slide 39) and the **Reciprocal-Space Plane-Wave Spectrum** $|c(\mathbf{G})|^2$. See how removing the core oscillations inside $r < r_c$ causes the high-$G$ Fourier components to decay orders of magnitude faster!


**Key Takeaway:** Outside $r \ge r_c$ (the bonding region), $\Psi_{\text{pseudo}}(r)$ and $\Psi_{\text{AE}}(r)$ are **identical**. Increasing $r_c$ makes the pseudopotential "softer" (requiring fewer plane waves and a lower kinetic cutoff $E_{\text{cut}} = \frac{\hbar^2}{2m}G_{\max}^2$), whereas a smaller $r_c$ is "harder" (higher $E_{\text{cut}}$) but more transferable across shortened high-pressure bonds.


In [43]:
ws.render_section('q34_dft_in_practice_real_vs_reciprocal')

**Q34 · medium** · 💡

In a modern plane-wave DFT code using Fast Fourier Transforms (FFTs, `05.QuantMech.pdf`, p. 37), which energy terms are evaluated in **reciprocal ($\mathbf{G}$) space** and which are evaluated in **real ($\mathbf{r}$) space**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the sub-bullets under Step 3 on Slide 37 ('Calculate total energy for trial orbitals').

</div>
</details>

- [ ] **(A)** Kinetic energy and Hartree electrostatic energy are evaluated in reciprocal space (where nabla^2 and Poisson's equation are diagonal in G), while exchange-correlation and local external potentials are evaluated in real space.
- [ ] **(B)** All terms are evaluated exclusively in real space without using reciprocal space.
- [ ] **(C)** Kinetic energy is evaluated in real space while LDA exchange-correlation is evaluated analytically in reciprocal space.
- [ ] **(D)** Fast Fourier Transforms scale as O(N^4) and are avoided in plane-wave codes.

In [44]:
ws.render_section('q35_al_pseudopotential_valence_count_numeric')

**Q35 · easy** · 💡

As shown on Slide 38 (`05.QuantMech.pdf`), an all-electron Aluminum atom ($\text{Al}, Z = 13$) has core states $1s^2$ ($-1512\text{ eV}$), $2s^2$ ($-108\text{ eV}$), and $2p^6$ ($-69.8\text{ eV}$), and valence states $3s^2$ ($-7.8\text{ eV}$) and $3p^1$ ($-2.7\text{ eV}$). When the inner shells are frozen into a pseudopotential ("Pseudo Al"), what is the effective valence ionic charge $Z_{\text{pseudo}}$ (number of active valence electrons per Al atom)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Subtract the 10 frozen core electrons (1s^2 2s^2 2p^6) from Z = 13, or read 'Pseudo Al Z = ...' on Slide 38.

</div>
</details>

_Numeric answer required._

In [45]:
ws.render_section('q36_pseudopotential_matching_outside_rc')

**Q36 · easy** · 💡

According to Slide 39 (`05.QuantMech.pdf`), how do the pseudo-wavefunction $\Psi_{\text{pseudo}}(r)$ and pseudopotential $V_{\text{pseudo}}(r)$ compare to the all-electron wavefunction $\Psi_{\text{AE}}(r)$ and Coulomb potential $V(r) \sim -Z/r$ **inside** ($r < r_c$) versus **outside** ($r \ge r_c$) the core cutoff radius $r_c$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Examine the red (pseudo) and dashed blue (real) curves on Slide 39 inside and outside the vertical black line r_c.

</div>
</details>

- [ ] **(A)** Inside r < r_c, V_pseudo is smooth/finite and Psi_pseudo is smooth and nodeless (removing core oscillations); outside r >= r_c, Psi_pseudo(r) and V_pseudo(r) are identical to the true all-electron wavefunction and potential.
- [ ] **(B)** Outside r >= r_c, Psi_pseudo(r) is set to zero while inside r < r_c it oscillates faster than Psi_AE(r).
- [ ] **(C)** V_pseudo(r) diverges as -Z^2/r^3 at r -> 0.
- [ ] **(D)** Psi_pseudo(r) has twice as many radial nodes inside r < r_c as Psi_AE(r).

In [46]:
ws.render_section('q37_nonlocal_pseudopotential_orthogonality')

**Q37 · medium** · 💡

Why do **non-local pseudopotentials** (Slide 40) require different effective radial potentials $V_l(r)$ for different angular momentum channels ($s, p, d$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the two bullet points on Slide 40 regarding states that have a core shell below them with the same angular momentum l.

</div>
</details>

- [ ] **(A)** Electron states with different angular momenta l scatter differently from the core; specifically, valence states that have a core shell below them of the SAME angular momentum l feel stronger effective repulsion due to the Pauli orthogonality constraint.
- [ ] **(B)** Because s, p, and d electrons have different electric charges.
- [ ] **(C)** Because only d-electrons experience nuclear attraction outside r_c.
- [ ] **(D)** Because spherical harmonics Y_{lm} are not orthogonal to one another.

In [47]:
ws.render_section('q38_basis_set_molecules_vs_solids')

**Q38 · easy** · 💡

According to Slide 41 (`05.QuantMech.pdf`), which basis set is most natural for **isolated molecules** versus **periodic crystalline solids**, and how do we systematically converge a basis-set calculation toward completeness?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the bullet points under 'Basis set choices' and 'Convergence' on Slide 41.

</div>
</details>

- [ ] **(A)** Localized atomic orbitals or Gaussians are typically used for isolated molecules, whereas periodic plane waves (sines and cosines) with Bloch's theorem and k-point sampling are used for periodic solids; convergence is achieved by systematically increasing the basis set size (e.g. G_max / E_cut) and k-point mesh.
- [ ] **(B)** Single plane waves are used for isolated molecules in vacuum, whereas non-periodic polynomials are used for bulk crystals.
- [ ] **(C)** Reducing the basis set size to a single function per unit cell guarantees exact completeness.
- [ ] **(D)** k-point sampling is only required for isolated gas-phase atoms.

In [48]:
ws.render_section('sec_bloch_band_structure_and_dft_failures')

### Bloch's Theorem and Reciprocal Space (Slides 41–43)
In a periodic crystal defined by Bravais lattice vectors $\{\mathbf{a}_1, \mathbf{a}_2, \mathbf{a}_3\}$, the Hamiltonian commutes with lattice translations ($[\hat{\mathcal{H}}, \hat{\mathcal{T}}_{\mathbf{R}}] = 0$). By **Bloch's Theorem**, the eigenstates take the form of a plane wave modulated by a lattice-periodic function $u_{n\mathbf{k}}(\mathbf{r} + \mathbf{R}) = u_{n\mathbf{k}}(\mathbf{r})$:
$$\boxed{\varphi_{n\mathbf{k}}(\mathbf{r}) = u_{n\mathbf{k}}(\mathbf{r})\exp(i\mathbf{k}\cdot\mathbf{r}) \quad \Longleftrightarrow \quad \psi(\mathbf{r} + \mathbf{R}) = \psi(\mathbf{r})\exp(i\mathbf{k}\cdot\mathbf{R})}$$
Expanding $u_{n\mathbf{k}}(\mathbf{r}) = \sum_{|\mathbf{G}| \le G_{\max}} c_{n\mathbf{k}}(\mathbf{G})\exp(i\mathbf{G}\cdot\mathbf{r})$ over reciprocal lattice vectors $\mathbf{G}$ generated by:
$$\mathbf{b}_k = 2\pi \frac{\mathbf{a}_l \times \mathbf{a}_m}{\mathbf{a}_k \cdot (\mathbf{a}_l \times \mathbf{a}_m)}$$
allows us to restrict the crystal wavevector $\mathbf{k}$ to the **First Brillouin Zone** — the **Wigner–Seitz cell of the reciprocal lattice** (Slide 43).

### Physical Origin of the Electronic Band Gap: Bragg Diffraction Picture (Slides 44–45)
Why do energy gaps open in periodic crystals? Consider a 1D crystal with lattice constant $a$:
- An electron wave undergoes **Bragg diffraction** when $n\lambda = 2a \implies k = \frac{2\pi}{\lambda} = \pm \frac{n\pi}{a}$ (at the Brillouin-zone boundary $k = \pm\frac{\pi}{a}$ for $n=1$).
- At $k = \frac{\pi}{a}$, the forward-traveling incident wave $\psi_i = e^{i\pi x/a}$ and the backward-diffracted wave $\psi_r = e^{-i\pi x/a}$ are degenerate and mix equally to form **two standing waves** (no traveling wave solution exists!):
  $$\psi_+(x) = \psi_i + \psi_r = 2\cos\left(\frac{\pi x}{a}\right), \qquad \psi_-(x) = \psi_i - \psi_r = 2i\sin\left(\frac{\pi x}{a}\right)$$
- **Energy Splitting $\Delta E = E_{\text{gap}} = 2U$ (Slide 45):**
  - $|\psi_+(x)|^2 \propto \cos^2(\pi x/a)$ piles electron probability density **directly on top of the positive ion cores** ($x = 0, \pm a, \pm 2a, \dots$), lowering its potential energy by $-U$: $E_+ = E_{\text{free}} - U$.
  - $|\psi_-(x)|^2 \propto \sin^2(\pi x/a)$ piles electron density **in between the ions** ($x = \pm\frac{a}{2}, \pm\frac{3a}{2}, \dots$), raising its energy by $+U$: $E_- = E_{\text{free}} + U$.
  - Thus, interaction with the periodic ion potential $U$ opens a **band gap** at $k = \pm\frac{\pi}{a}$:
    $$\boxed{E_{\text{gap}} = E_- - E_+ = (E_{\text{free}} + U) - (E_{\text{free}} - U) = 2U}$$
  - Near the diffraction condition ($k \to \pm\pi/a$), the band dispersion $E(k)$ curves parabolically ($\frac{dE}{dk} = 0$ at the band edges), so electrons behave as "free" quasiparticles with a modified **effective mass** $m^* = \hbar^2\left(\frac{d^2E}{dk^2}\right)^{-1}$!

### Metallic vs. Semiconducting Band Structures: $\text{Cu}$ vs. $\text{Si}$ (Slides 46–47)
- In **Copper ($\text{Cu}$, Slide 47)**, a wide, nearly-free-electron $4s$ band is intersected at intermediate energies ($-2\text{ to }-5\text{ eV}$ below $E_F$) by narrow, atomic-like $3d$ bands, leading to **$s\text{–}d$ hybridization** and **necking of the Fermi surface along $\langle 111\rangle$ ($L$-point) directions** (which drives **Hume-Rothery alloy phase stabilization**!).

### Quantitative Successes & Failures of Kohn–Sham DFT (Slides 48–52)
1. **Structural & Elastic Properties (Slide 48):** Lattice parameters are typically accurate to **within $1\text{–}3\%$** (often better than $1\%$), and **bulk moduli / elastic constants** are within **$\sim 10\%$**.
2. **Vibrational / Phonon Properties (Slide 48):** Atomic **forces** ($\mathbf{F} = -\nabla E$, 1st derivatives) are accurate to **better than $10\%$** (similar to elastic constants). Because **vibrational phonon frequencies** $\omega \propto \sqrt{k}$ are proportional to the square root of the 2nd derivative of energy ($\frac{\delta\omega}{\omega} \approx \frac{1}{2}\frac{\delta k}{k}$), phonon frequencies have about **$1/2$ the relative error of forces — i.e., $\sim 5\%$ accuracy** (vastly outperforming empirical EAM potentials in $\text{Ni}$, Slide 48)!
3. **Point Defect Energies (Slide 49):** Vacancy and interstitial formation energies in metals are highly accurate (**within $\sim 0.1\text{ eV}$**), but for covalent **self-interstitial defects in Silicon**, DFT-LDA/GGA underestimates the formation energy by **$\sim 1\text{ eV}$** (predicting $3\text{–}3.5\text{ eV}$ vs. $4.5\text{ eV}$ from experiment and QMC).
4. **The DFT Band-Gap Problem (Slides 49–51):** Because Kohn–Sham DFT is strictly a **ground-state** theory and local/semi-local functionals (LDA/GGA) suffer from residual self-interaction error and lack the exact derivative discontinuity when adding an electron, **LDA and GGA underestimate semiconductor band gaps by $20\text{–}50\%$**!
   - In narrow-gap semiconductors like **$\text{Ge}$** ($E_{\text{exp}} = 0.74\text{ eV}$) and **$\text{InAs}$** ($E_{\text{exp}} = 0.41\text{ eV}$, where LDA gives $-0.42\text{ eV}$ and GGA-PBE gives $-0.13\text{ eV}$, Slide 50), **LDA and GGA falsely predict a metallic ground state (negative/zero band gap)!**
   - **Solutions:** **Hybrid functionals** (B3LYP, HSE06 gives $0.39\text{ eV}$ for $\text{InAs}$, $\sim 10\%$ accuracy) or many-body excited-state theories like the **$GW$ approximation** and **Quantum Monte Carlo (QMC)** (Slides 50–51).


In [49]:
ws.render_section('concept_bloch_bandgap_and_dft')

### Interactive Explorer: Brillouin-Zone Band Gap Opening & The DFT Band-Gap Problem

Adjust the periodic crystal potential strength $U$ (in $\text{eV}$) to watch Bragg diffraction open a band gap $E_{\text{gap}} = 2U$ at the 1st Brillouin-zone boundary $k = \pm\pi/a$, or switch to the **DFT vs. $GW$ Semiconductor Band-Gap Benchmark** (matching Slides 50–51).


**Key Takeaway:** At $k = \pm\pi/a$, Bragg reflection converts traveling waves into standing waves $\psi_+ \propto \cos(\pi x/a)$ (density on ions, energy $E_{\text{free}} - U$) and $\psi_- \propto \sin(\pi x/a)$ (density between ions, energy $E_{\text{free}} + U$), opening $E_{\text{gap}} = 2U$. In Kohn–Sham DFT, LDA/GGA underestimates $E_{\text{gap}}$ by $20\text{–}50\%$, turning narrow-gap semiconductors ($\text{InAs}, \text{Ge}$) into false metals unless hybrid functionals (HSE) or $GW$ corrections are applied.


In [50]:
ws.render_section('q39_bloch_theorem_and_brillouin_zone')

**Q39 · easy** · 💡

According to Slide 43 (`05.QuantMech.pdf`), what is **Bloch's Theorem** for an electron in a crystal with Bravais lattice translation vector $\mathbf{R}$, and how is the **First Brillouin Zone** geometrically defined?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the second and fifth bullet points on Slide 43.

</div>
</details>

- [ ] **(A)** Bloch's theorem states that psi(r + R) = psi(r) * exp(i k . R) (or phi(r) = u_nk(r) * exp(i k . r) with periodic u_nk); the Brillouin zone is the Wigner-Seitz primitive cell of the reciprocal lattice.
- [ ] **(B)** Bloch's theorem states that psi(r + R) = 0 for all lattice vectors R; the Brillouin zone is the real-space unit cell.
- [ ] **(C)** Bloch's theorem applies only to amorphous glasses and liquids.
- [ ] **(D)** The Brillouin zone is a sphere of radius 1 Bohr in real space.

In [51]:
ws.render_section('q40_reciprocal_lattice_1d_zone_boundary')

**Q40 · easy** · 💡

In a 1D crystal with real-space lattice spacing $a$ (Slides 44–45), the fundamental reciprocal lattice vector has magnitude $\Delta k = G_1 = \frac{2\pi}{a}$. At what wavevectors $k$ do the boundaries of the **First Brillouin Zone** (where lowest-order $n=1$ Bragg diffraction occurs) lie?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the horizontal axis of the E(k) plot on Slide 45 or the Bragg condition k = n*pi/a for n = +/- 1 on Slide 44.

</div>
</details>

- [ ] **(A)** k = -pi/a and k = +pi/a (so the total width of the 1st Brillouin zone is Delta k = 2*pi/a)
- [ ] **(B)** k = 0 and k = +4*pi/a
- [ ] **(C)** k = -a/pi and k = +a/pi
- [ ] **(D)** k = -1/a and k = +1/a

In [52]:
ws.render_section('q41_standing_waves_origin_of_gap')

**Q41 · medium** · 💡

At the Brillouin-zone boundary $k = \frac{\pi}{a}$ (Slides 44–45), the incident wave $\psi_i = e^{i\pi x/a}$ and reflected wave $\psi_r = e^{-i\pi x/a}$ superpose to form two standing waves: $\psi_+(x) = 2\cos(\pi x/a)$ and $\psi_-(x) = 2i\sin(\pi x/a)$. Why do these two states have different energies $E = E_{\text{free}} \mp U$ in the presence of the periodic ion potential $U$?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare where |cos(pi*x/a)|^2 peaks (at x = 0, a, 2a on the positive ions) versus where |sin(pi*x/a)|^2 peaks (at x = a/2, 3a/2 between the ions).

</div>
</details>

- [ ] **(A)** psi_+(x) = 2*cos(pi*x/a) concentrates electron density on the attractive ion cores (lowering energy to E_free - U), whereas psi_-(x) = 2i*sin(pi*x/a) concentrates electron density between the ions (raising energy to E_free + U), opening a band gap E_gap = 2U.
- [ ] **(B)** psi_+(x) and psi_-(x) have different free-electron kinetic energies hbar^2 k^2 / (2m).
- [ ] **(C)** psi_+(x) travels to the right at speed v_F while psi_-(x) travels to the left at speed 2*v_F.
- [ ] **(D)** Only psi_+(x) satisfies Bloch's theorem.

In [53]:
ws.render_section('q42_bragg_bandgap_numeric')

**Q42 · easy** · 💡

In the nearly-free-electron diffraction model on Slide 45 (`05.QuantMech.pdf`), if the free-electron kinetic energy at $k = \pi/a$ is $E_{\text{free}} = 4.20\text{ eV}$ and the periodic ion potential perturbation has amplitude $U = 0.65\text{ eV}$, what is the magnitude of the resulting energy band gap $E_{\text{gap}} = E_- - E_+$ at $k = \pi/a$ (in $\text{eV}$)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use E_gap = (E_free + U) - (E_free - U) = 2 * U from Slide 45.

</div>
</details>

_Numeric answer required._

In [54]:
ws.render_section('q43_copper_band_structure_features')

**Q43 · medium** · 💡

In the Wien2k DFT band structure and Fermi surface of **Copper ($\text{Cu}$)** on Slide 47 (`05.QuantMech.pdf`), which of the following physical features are observed? *(Select all that apply.)*


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the four bullet points on Slide 47 and inspect the yellow shaded band region.

</div>
</details>

- [ ] **(A)** A wide, nearly-free-electron 4s band dominates at low (-10 to -5 eV) and high (> -1 eV) energies.
- [ ] **(B)** Strong hybridization occurs between the nearly-free 4s band and narrow, atomic-like 3d bands at intermediate energies (-5 to -2 eV, yellow shaded region).
- [ ] **(C)** The Fermi surface develops 'necks' touching the Brillouin-zone boundary along the [111] (L-point) directions, which drives Hume-Rothery alloy phase stabilization.
- [ ] **(D)** Copper has a 5.0 eV insulating band gap at the Fermi level E_F = 0.

In [55]:
ws.render_section('q44_dft_structural_and_phonon_accuracy')

**Q44 · medium** · 💡

According to Slide 48 (`05.QuantMech.pdf`), standard Kohn–Sham DFT predicts atomic **forces** (1st derivatives of energy) and **elastic constants / bulk moduli** (2nd derivatives) to within $\sim 10\%$. Why are DFT **vibrational phonon frequencies** typically accurate to **$\sim 5\%$** (about $1/2$ the relative error of the forces/force constants)?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall that harmonic vibrational frequency scales as the square root of the 2nd-derivative force constant: omega = sqrt(k / M). Propagate a 10% relative error in k through the square root!

</div>
</details>

- [ ] **(A)** Because vibrational frequencies omega = sqrt(k / M) scale as the square root of the 2nd-derivative force constants k, so a ~10% relative error in k (and forces) produces a relative frequency error delta_omega / omega approx (1/2) * (delta_k / k) approx 5%.
- [ ] **(B)** Because phonon frequencies depend only on nuclear masses and not on the interatomic potential energy.
- [ ] **(C)** Because DFT uses empirical EAM force constants to compute phonon dispersions.
- [ ] **(D)** Because the 2nd derivative of any DFT energy functional is zero.

In [56]:
ws.render_section('q45_dft_defect_energies_silicon_interstitial')

**Q45 · medium** · 💡

According to Slide 49 (`05.QuantMech.pdf`), how accurate is standard DFT (LDA/GGA) for point defect formation energies in **metals** versus **self-interstitial defects in Silicon**?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Read the two bullet points under 'Defect energies' on Slide 49.

</div>
</details>

- [ ] **(A)** In metals, vacancy and interstitial formation energies are highly accurate (within ~0.1 eV); however, for self-interstitials in silicon, DFT is too low by ~1 eV (predicting 3-3.5 eV instead of the 4.5 eV from experiment and QMC).
- [ ] **(B)** In metals, DFT defect energies are off by 10 eV, whereas in silicon they are exact to 0.001 eV.
- [ ] **(C)** DFT always predicts negative formation energies for all vacancies in metals.
- [ ] **(D)** Quantum Monte Carlo (QMC) cannot be applied to point defects in silicon.

In [57]:
ws.render_section('q46_dft_bandgap_problem_inas_and_ge')

**Q46 · medium** · 💡

As demonstrated on Slides 49–51 (`05.QuantMech.pdf`) for **$\text{InAs}$** (experimental band gap $0.41\text{ eV}$) and **$\text{Ge}$** (experimental band gap $0.74\text{ eV}$), what is the **"band-gap problem" of DFT**, and which computational methods resolve it?


<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compare the LDA (-0.42 eV), PBE (-0.13 eV), B3LYP/HSE (0.39-0.54 eV), and GW band gaps on Slides 50-51.

</div>
</details>

- [ ] **(A)** Standard Kohn-Sham LDA and GGA fail for excited states and underestimate semiconductor band gaps by 20-50%, falsely predicting narrow-gap semiconductors like InAs and Ge to be metallic (zero/negative gap); Hybrid functionals (B3LYP, HSE) and the GW approximation / QMC restore accurate band gaps.
- [ ] **(B)** LDA and GGA overestimate the band gap of InAs as 4.1 eV instead of 0.41 eV.
- [ ] **(C)** The GW approximation predicts all semiconductors and insulators (including Diamond and MgO) to be metals.
- [ ] **(D)** Adding a larger plane-wave cutoff E_cut in LDA fixes the InAs band gap to 0.41 eV.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()